<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/02%20bigquery/07_ENARES_2024_STAGE2_INTEGRATED_CLOSURE_CHECK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07_ENARES_2024_STAGE2_INTEGRATED_CLOSURE_CHECK.ipynb

**Project:** ENARES 2024 - CRS04 Module  
**Stage:** 02  
**Purpose:** perform the final integrated closure validation for the Stage 02 Sprint and Competency Annex.

## Issues covered

- **Issue #19:** Cloud security and network security validation.
- **Issue #20:** Integrated sprint and competency closure check.

## Methodological Boundary

This notebook does not ingest, merge, recode, derive variables, create indicators, run statistical models, or interpret substantive results.

It only verifies that the required Stage 02 evidence files exist and that the security documentation contains the required controls.

In [ ]:
# ============================================================
# 0. Setup
# ============================================================

!pip install -q pandas

from google.colab import drive
from datetime import datetime, timezone
import pandas as pd
import os
import glob

drive.mount("/content/drive")

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"
DOCS_DIR = f"{ROOT_DRIVE}/docs"

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()

print("ROOT_DRIVE:", ROOT_DRIVE)
print("LOG_DIR:", LOG_DIR)
print("DOCS_DIR:", DOCS_DIR)
print("RUN_UTC:", RUN_UTC)
print("Notebook 07 initialized.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT_DRIVE: /content/drive/MyDrive/ENARES_2024_PROJECT
LOG_DIR: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs
DOCS_DIR: /content/drive/MyDrive/ENARES_2024_PROJECT/docs
RUN_UTC: 2026-06-19T23:56:45.570346+00:00
Notebook 07 initialized.


## 1. Stage 02 Closure Policy

This block records the allowed and forbidden actions for the final closure notebook.

In [ ]:
closure_policy = pd.DataFrame([{
    "stage": "STAGE02",
    "notebook": "07_ENARES_2024_STAGE2_INTEGRATED_CLOSURE_CHECK.ipynb",
    "purpose": "Final integrated closure validation for Stage 02 Sprint and Competency Annex",
    "allowed_actions": "closure_reporting; security_validation; documentation_validation; artifact_existence_check",
    "forbidden_actions": "merge; recode; derived_variables; indicators; statistical_models; substantive_interpretation",
    "raw_data_write_operations": False,
    "checked_at_utc": RUN_UTC,
}])

closure_policy_output = f"{LOG_DIR}/ENARES_2024_STAGE2_INTEGRATED_CLOSURE_POLICY.csv"
closure_policy.to_csv(closure_policy_output, index=False)

display(closure_policy)
print("Saved:", closure_policy_output)

,stage,notebook,purpose,allowed_actions,forbidden_actions,raw_data_write_operations,checked_at_utc
0,STAGE02,07_ENARES_2024_STAGE2_INTEGRATED_CLOSURE_CHECK...,Final integrated closure validation for Stage ...,closure_reporting; security_validation; docume...,merge; recode; derived_variables; indicators; ...,False,2026-06-19T23:56:45.570346+00:00


Saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_INTEGRATED_CLOSURE_POLICY.csv


## 2. Issue #19 — Security Validation

This block checks whether the security architecture document exists and documents the required security controls:

- OAuth 2.0
- Service Account
- Least Privilege IAM
- HTTPS / TLS
- Credentials outside notebooks and repositories
- Microdata protection
- Governance and auditability

In [ ]:
# ============================================================
# 2. Issue #19 - Security Validation
# ============================================================

possible_security_docs = [
    f"{DOCS_DIR}/ENARES_2024_STAGE2_SECURITY_ARCHITECTURE.md",
    f"{DOCS_DIR}/ENARES_2024_STAGE2_security_architecture.md",
]

security_doc = next((path for path in possible_security_docs if os.path.exists(path)), None)

security_doc_text = ""
if security_doc:
    with open(security_doc, "r", encoding="utf-8") as f:
        security_doc_text = f.read().lower()

security_controls = [
    {
        "control": "OAuth 2.0",
        "accepted_terms": ["oauth"],
    },
    {
        "control": "Service Account",
        "accepted_terms": ["service account", "service accounts"],
    },
    {
        "control": "Least Privilege IAM",
        "accepted_terms": [
            "least privilege",
            "mínimo privilegio",
            "minimo privilegio",
            "iam",
        ],
    },
    {
        "control": "HTTPS/TLS",
        "accepted_terms": ["https", "tls"],
    },
    {
        "control": "Credentials not persisted in code",
        "accepted_terms": [
            "credential",
            "credentials",
            "credencial",
            "credenciales",
        ],
    },
    {
        "control": "Microdata protection",
        "accepted_terms": [
            "microdata",
            "microdatos",
            "sensitive information",
            "información sensible",
            "informacion sensible",
        ],
    },
    {
        "control": "Governance and auditability",
        "accepted_terms": [
            "governance",
            "gobernanza",
            "audit",
            "auditability",
            "auditoría",
            "auditoria",
            "trazabilidad",
        ],
    },
]

security_rows = []

for item in security_controls:
    terms = item["accepted_terms"]

    documented = bool(security_doc_text) and any(
        term.lower() in security_doc_text
        for term in terms
    )

    security_rows.append({
        "control": item["control"],
        "required": True,
        "documented": documented,
        "accepted_terms": "; ".join(terms),
        "evidence_document": security_doc,
        "checked_at_utc": RUN_UTC,
    })

security_validation = pd.DataFrame(security_rows)

security_validation_output = f"{LOG_DIR}/ENARES_2024_STAGE2_SECURITY_VALIDATION.csv"
security_validation.to_csv(security_validation_output, index=False)

display(security_validation)

if security_doc is None:
    raise FileNotFoundError(
        "Security architecture document was not found. Expected one of: "
        + ", ".join(possible_security_docs)
    )

if not security_validation["documented"].all():
    missing_controls = security_validation.loc[
        ~security_validation["documented"],
        "control"
    ].tolist()

    raise AssertionError(
        "Missing documented security controls: "
        + ", ".join(missing_controls)
    )

print("Issue #19 security validation PASS")
print("Saved:", security_validation_output)

,control,required,documented,accepted_terms,evidence_document,checked_at_utc
0,OAuth 2.0,True,True,oauth,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,2026-06-19T23:56:45.570346+00:00
1,Service Account,True,True,service account; service accounts,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,2026-06-19T23:56:45.570346+00:00
2,Least Privilege IAM,True,True,least privilege; mínimo privilegio; minimo pri...,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,2026-06-19T23:56:45.570346+00:00
3,HTTPS/TLS,True,True,https; tls,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,2026-06-19T23:56:45.570346+00:00
4,Credentials not persisted in code,True,True,credential; credentials; credencial; credenciales,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,2026-06-19T23:56:45.570346+00:00
5,Microdata protection,True,True,microdata; microdatos; sensitive information; ...,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,2026-06-19T23:56:45.570346+00:00
6,Governance and auditability,True,True,governance; gobernanza; audit; auditability; a...,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,2026-06-19T23:56:45.570346+00:00


Issue #19 security validation PASS
Saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_SECURITY_VALIDATION.csv


## 3. Expected Stage 02 Artifacts

This block defines all required Stage 02 artifacts for the integrated annex closure.

It includes canonical Stage 02 outputs, competency evidence, documentation, profiling reports, observability, lineage, and security validation.

In [ ]:
# ============================================================
# 3. Expected Stage 02 Artifacts
# ============================================================

def first_existing(candidates):
    for path in candidates:
        if os.path.exists(path):
            return path
    return candidates[0]

expected_artifacts = []

# ------------------------------------------------------------
# Canonical Stage 02 outputs
# ------------------------------------------------------------
canonical_outputs = [
    {
        "issue": "Stage 02 Core",
        "category": "BigQuery setup",
        "filename": "ENARES_2024_STAGE2_bigquery_dataset_registry.csv",
        "path": f"{LOG_DIR}/ENARES_2024_STAGE2_bigquery_dataset_registry.csv",
        "required": True,
    },
    {
        "issue": "Stage 02 Core",
        "category": "Raw ingestion",
        "filename": "ENARES_2024_STAGE2_source_file_check.csv",
        "path": f"{LOG_DIR}/ENARES_2024_STAGE2_source_file_check.csv",
        "required": True,
    },
    {
        "issue": "Stage 02 Core",
        "category": "Raw ingestion",
        "filename": "ENARES_2024_STAGE2_raw_table_inventory.csv",
        "path": f"{LOG_DIR}/ENARES_2024_STAGE2_raw_table_inventory.csv",
        "required": True,
    },
    {
        "issue": "Stage 02 Core",
        "category": "Raw ingestion",
        "filename": "ENARES_2024_STAGE2_rowcount_validation.csv",
        "path": f"{LOG_DIR}/ENARES_2024_STAGE2_rowcount_validation.csv",
        "required": True,
    },
    {
        "issue": "Stage 02 Core",
        "category": "Metadata",
        "filename": "ENARES_2024_STAGE2_metadata_inventory.csv",
        "path": f"{LOG_DIR}/ENARES_2024_STAGE2_metadata_inventory.csv",
        "required": True,
    },
    {
        "issue": "Stage 02 Core",
        "category": "Metadata",
        "filename": "ENARES_2024_STAGE2_metadata_source_files.csv",
        "path": f"{LOG_DIR}/ENARES_2024_STAGE2_metadata_source_files.csv",
        "required": True,
    },
    {
        "issue": "Stage 02 Core",
        "category": "Cloud storage report",
        "filename": "ENARES_2024_STAGE2_cloud_storage_report.md",
        "path": f"{LOG_DIR}/ENARES_2024_STAGE2_cloud_storage_report.md",
        "required": True,
    },
]

expected_artifacts.extend(canonical_outputs)

# ------------------------------------------------------------
# Issue #12 - ERD / Logical and Physical Data Model
# Accept English or original filename.
# ------------------------------------------------------------
erd_path = first_existing([
    f"{DOCS_DIR}/ENARES_2024_STAGE2_CRS04_LOGICAL_PHYSICAL_DATA_MODEL.md",
    f"{DOCS_DIR}/ENARES_2024_STAGE2_CRS04_ERD.md",
])

expected_artifacts.append({
    "issue": "#12",
    "category": "Logical and Physical Data Model",
    "filename": os.path.basename(erd_path),
    "path": erd_path,
    "required": True,
})

# ------------------------------------------------------------
# Issue #13 - Partitioning Strategy
# Accept English or original filename.
# ------------------------------------------------------------
partitioning_path = first_existing([
    f"{DOCS_DIR}/ENARES_2024_STAGE2_PARTITIONING_STRATEGY.md",
    f"{DOCS_DIR}/ENARES_2024_STAGE2_partitioning_strategy.md",
])

expected_artifacts.append({
    "issue": "#13",
    "category": "Partitioning Strategy",
    "filename": os.path.basename(partitioning_path),
    "path": partitioning_path,
    "required": True,
})

# ------------------------------------------------------------
# Issue #14 - Data Profiling
# Accept English or original filenames.
# ------------------------------------------------------------
profile_tables = [
    "raw_crs04_cap100",
    "raw_crs04_cap200",
    "raw_crs04_cap248",
    "raw_crs04_cap300",
]

for table_name in profile_tables:
    upper_suffix = table_name.upper()
    profile_path = first_existing([
        f"{LOG_DIR}/ENARES_2024_STAGE2_DATA_PROFILE_{upper_suffix}.html",
        f"{LOG_DIR}/ENARES_2024_STAGE2_profile_{table_name}.html",
    ])

    expected_artifacts.append({
        "issue": "#14",
        "category": "Data Profiling HTML",
        "filename": os.path.basename(profile_path),
        "path": profile_path,
        "required": True,
    })

profiling_validation_path = first_existing([
    f"{LOG_DIR}/ENARES_2024_STAGE2_DATA_PROFILING_VALIDATION.csv",
    f"{LOG_DIR}/ENARES_2024_STAGE2_profiling_validation.csv",
])

expected_artifacts.append({
    "issue": "#14",
    "category": "Data Profiling Validation",
    "filename": os.path.basename(profiling_validation_path),
    "path": profiling_validation_path,
    "required": True,
})

# ------------------------------------------------------------
# Issue #15 - Security Architecture
# ------------------------------------------------------------
security_doc_path = first_existing([
    f"{DOCS_DIR}/ENARES_2024_STAGE2_SECURITY_ARCHITECTURE.md",
    f"{DOCS_DIR}/ENARES_2024_STAGE2_security_architecture.md",
])

expected_artifacts.append({
    "issue": "#15",
    "category": "Security Architecture",
    "filename": os.path.basename(security_doc_path),
    "path": security_doc_path,
    "required": True,
})

# ------------------------------------------------------------
# Issues #16-#18 - Competency checks
# ------------------------------------------------------------
competency_outputs = [
    ("#16", "Layered Dataset Architecture", "ENARES_2024_STAGE2_competency_dataset_layers_check.csv"),
    ("#17", "Raw Load Validation", "ENARES_2024_STAGE2_competency_raw_load_check.csv"),
    ("#18", "SPSS Metadata Validation", "ENARES_2024_STAGE2_competency_metadata_check.csv"),
    ("#18", "PDF Metadata Validation", "ENARES_2024_STAGE2_competency_pdf_metadata_check.csv"),
    ("#18", "PDF Metadata Schema Validation", "ENARES_2024_STAGE2_competency_pdf_metadata_schema_check.csv"),
    ("#18", "Competency Partial Closure", "ENARES_2024_STAGE2_competency_16_18_closure_check.csv"),
]

for issue, category, filename in competency_outputs:
    expected_artifacts.append({
        "issue": issue,
        "category": category,
        "filename": filename,
        "path": f"{LOG_DIR}/{filename}",
        "required": True,
    })

# ------------------------------------------------------------
# Observability and lineage
# ------------------------------------------------------------
observability_outputs = [
    ("Observability", "Ingestion Observability", "ENARES_2024_STAGE2_ingestion_observability.csv"),
    ("Lineage", "Source File Lineage", "ENARES_2024_STAGE2_lineage_source_files.csv"),
]

for issue, category, filename in observability_outputs:
    expected_artifacts.append({
        "issue": issue,
        "category": category,
        "filename": filename,
        "path": f"{LOG_DIR}/{filename}",
        "required": True,
    })

# ------------------------------------------------------------
# Issue #19 - Security validation
# ------------------------------------------------------------
expected_artifacts.append({
    "issue": "#19",
    "category": "Security Validation",
    "filename": "ENARES_2024_STAGE2_SECURITY_VALIDATION.csv",
    "path": security_validation_output,
    "required": True,
})

# ------------------------------------------------------------
# Notebook 06 closure output
# Accept English or original filename.
# ------------------------------------------------------------
notebook06_closure_path = first_existing([
    f"{LOG_DIR}/ENARES_2024_STAGE2_DOCUMENTATION_AND_PROFILING_CLOSURE_CHECK.csv",
    f"{LOG_DIR}/ENARES_2024_STAGE2_documentation_profiling_closure_check.csv",
])

expected_artifacts.append({
    "issue": "Notebook 06",
    "category": "Documentation and Profiling Closure",
    "filename": os.path.basename(notebook06_closure_path),
    "path": notebook06_closure_path,
    "required": True,
})

## 4. Issue #20 — Integrated Sprint and Competency Closure Check

This block verifies that all required Stage 02 annex artifacts exist.

In [ ]:
# ============================================================
# 4. Issue #20 - Integrated Closure Check
# ============================================================

closure_rows = []

for artifact in expected_artifacts:
    path = artifact["path"]
    exists = os.path.exists(path)
    size_bytes = os.path.getsize(path) if exists else None

    closure_rows.append({
        "issue": artifact["issue"],
        "category": artifact["category"],
        "filename": artifact["filename"],
        "path": path,
        "required": artifact["required"],
        "exists": exists,
        "size_bytes": size_bytes,
        "checked_at_utc": RUN_UTC,
    })

integrated_closure = pd.DataFrame(closure_rows)

integrated_closure["artifact_pass"] = (
    integrated_closure["exists"]
    | (~integrated_closure["required"])
)

stage2_integrated_annex_pass = bool(integrated_closure["artifact_pass"].all())

integrated_closure["stage2_integrated_annex_pass"] = stage2_integrated_annex_pass

integrated_closure_output = (
    f"{LOG_DIR}/ENARES_2024_STAGE2_INTEGRATED_SPRINT_COMPETENCY_CLOSURE_CHECK.csv"
)

integrated_closure.to_csv(integrated_closure_output, index=False)

display(integrated_closure)

if not stage2_integrated_annex_pass:
    missing = integrated_closure.loc[
        ~integrated_closure["artifact_pass"],
        ["issue", "category", "filename", "path"]
    ]
    display(missing)
    raise FileNotFoundError(
        "Stage 02 integrated annex closure failed. Missing required artifacts are displayed above."
    )

print("Issue #20 integrated closure PASS")
print("stage2_integrated_annex_pass:", stage2_integrated_annex_pass)
print("Saved:", integrated_closure_output)

,issue,category,filename,path,required,exists,size_bytes,checked_at_utc,artifact_pass,stage2_integrated_annex_pass
0,Stage 02 Core,BigQuery setup,ENARES_2024_STAGE2_bigquery_dataset_registry.csv,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,True,615,2026-06-19T23:56:45.570346+00:00,True,True
1,Stage 02 Core,Raw ingestion,ENARES_2024_STAGE2_source_file_check.csv,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,True,915,2026-06-19T23:56:45.570346+00:00,True,True
2,Stage 02 Core,Raw ingestion,ENARES_2024_STAGE2_raw_table_inventory.csv,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,True,911,2026-06-19T23:56:45.570346+00:00,True,True
3,Stage 02 Core,Raw ingestion,ENARES_2024_STAGE2_rowcount_validation.csv,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,True,946,2026-06-19T23:56:45.570346+00:00,True,True
4,Stage 02 Core,Metadata,ENARES_2024_STAGE2_metadata_inventory.csv,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,True,823,2026-06-19T23:56:45.570346+00:00,True,True
5,Stage 02 Core,Metadata,ENARES_2024_STAGE2_metadata_source_files.csv,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,True,7535,2026-06-19T23:56:45.570346+00:00,True,True
6,Stage 02 Core,Cloud storage report,ENARES_2024_STAGE2_cloud_storage_report.md,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,True,3909,2026-06-19T23:56:45.570346+00:00,True,True
7,#12,Logical and Physical Data Model,ENARES_2024_STAGE2_CRS04_ERD.md,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,True,True,1911,2026-06-19T23:56:45.570346+00:00,True,True
8,#13,Partitioning Strategy,ENARES_2024_STAGE2_partitioning_strategy.md,/content/drive/MyDrive/ENARES_2024_PROJECT/doc...,True,True,1850,2026-06-19T23:56:45.570346+00:00,True,True
9,#14,Data Profiling HTML,ENARES_2024_STAGE2_profile_raw_crs04_cap100.html,/content/drive/MyDrive/ENARES_2024_PROJECT/05R...,True,True,11621437,2026-06-19T23:56:45.570346+00:00,True,True


Issue #20 integrated closure PASS
stage2_integrated_annex_pass: True
Saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_INTEGRATED_SPRINT_COMPETENCY_CLOSURE_CHECK.csv


## 5. Executive Closure Summary

This block creates a concise summary table for reporting and defense.

In [ ]:
# ============================================================
# 5. Executive Closure Summary
# ============================================================

summary_rows = []

issue_groups = [
    ("#12", "Logical and Physical Data Model / ERD"),
    ("#13", "Partitioning Strategy"),
    ("#14", "Read-only Data Profiling"),
    ("#15", "Security Architecture"),
    ("#16", "Layered Dataset Architecture"),
    ("#17", "Raw Load and Traceability"),
    ("#18", "Metadata and Governance"),
    ("#19", "Cloud Security and Network Security Validation"),
    ("#20", "Integrated Sprint and Competency Closure"),
]

for issue, description in issue_groups:
    if issue == "#20":
        issue_pass = stage2_integrated_annex_pass
        evidence_count = len(integrated_closure)
    elif issue == "#19":
        subset = integrated_closure[integrated_closure["issue"] == issue]
        issue_pass = bool(subset["artifact_pass"].all()) and bool(security_validation["documented"].all())
        evidence_count = len(subset)
    else:
        subset = integrated_closure[integrated_closure["issue"] == issue]
        issue_pass = bool(subset["artifact_pass"].all()) if len(subset) else False
        evidence_count = len(subset)

    summary_rows.append({
        "issue": issue,
        "description": description,
        "evidence_count": evidence_count,
        "pass": issue_pass,
        "checked_at_utc": RUN_UTC,
    })

executive_summary = pd.DataFrame(summary_rows)

executive_summary_output = f"{LOG_DIR}/ENARES_2024_STAGE2_EXECUTIVE_CLOSURE_SUMMARY.csv"
executive_summary.to_csv(executive_summary_output, index=False)

display(executive_summary)

if not executive_summary["pass"].all():
    failed = executive_summary.loc[~executive_summary["pass"], "issue"].tolist()
    raise AssertionError("Some Stage 02 annex issues did not pass closure: " + ", ".join(failed))

print("Executive closure summary PASS")
print("Saved:", executive_summary_output)

,issue,description,evidence_count,pass,checked_at_utc
0,#12,Logical and Physical Data Model / ERD,1,True,2026-06-19T23:56:45.570346+00:00
1,#13,Partitioning Strategy,1,True,2026-06-19T23:56:45.570346+00:00
2,#14,Read-only Data Profiling,5,True,2026-06-19T23:56:45.570346+00:00
3,#15,Security Architecture,1,True,2026-06-19T23:56:45.570346+00:00
4,#16,Layered Dataset Architecture,1,True,2026-06-19T23:56:45.570346+00:00
5,#17,Raw Load and Traceability,1,True,2026-06-19T23:56:45.570346+00:00
6,#18,Metadata and Governance,4,True,2026-06-19T23:56:45.570346+00:00
7,#19,Cloud Security and Network Security Validation,1,True,2026-06-19T23:56:45.570346+00:00
8,#20,Integrated Sprint and Competency Closure,25,True,2026-06-19T23:56:45.570346+00:00


Executive closure summary PASS
Saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_EXECUTIVE_CLOSURE_SUMMARY.csv


## 6. Final Defense Statement

Use this statement in the final Stage 02 report or presentation:

> Stage 02 focused on raw data ingestion, metadata preservation, governance documentation, lineage verification, observability, security documentation, and read-only data profiling. No analytical transformations, recoding, derived variables, indicators, statistical modeling, or substantive interpretation were performed during this stage. The integrated closure check confirms that the Sprint and Competency Annex requirements for Issues #12 through #20 have been validated with auditable artifacts.

In [ ]:
print("FINAL STATUS")
print("------------")
print("Stage 02 Integrated Annex Pass:", stage2_integrated_annex_pass)
print("Security Validation Pass:", bool(security_validation["documented"].all()))
print("Executive Summary Pass:", bool(executive_summary["pass"].all()))
print("")
print("Generated outputs:")
print(" -", closure_policy_output)
print(" -", security_validation_output)
print(" -", integrated_closure_output)
print(" -", executive_summary_output)
print("")
print("Notebook 07 completed successfully.")

FINAL STATUS
------------
Stage 02 Integrated Annex Pass: True
Security Validation Pass: True
Executive Summary Pass: True

Generated outputs:
 - /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_INTEGRATED_CLOSURE_POLICY.csv
 - /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_SECURITY_VALIDATION.csv
 - /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_INTEGRATED_SPRINT_COMPETENCY_CLOSURE_CHECK.csv
 - /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_EXECUTIVE_CLOSURE_SUMMARY.csv

Notebook 07 completed successfully.
